<a href="https://colab.research.google.com/github/jysim128/BMACE-MIREX/blob/main/section_count.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Count Trap Sections


In [ ]:
import librosa
import pandas as pd
import os
import matplotlib.pyplot as plt
import glob
from google.colab import drive
import re
drive.mount('/content/drive')

TRAP_LABEL_DIR = "/content/drive/MyDrive/1. separated audio/trap_labels"
FORMAL_LABEL_DIR = "/content/drive/MyDrive/1. separated audio/unified_labels"
AUDIO_FILES_DIR = "/content/drive/MyDrive/1. separated audio/audio"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def merge_formal_trap(formal_dir, trap_dir, verbose=True):
    """formal 라벨(start, end, label) 옆에 trap 라벨의 label을 trap_label 열로 붙여 반환."""

    def load(directory):
        paths = sorted(glob.glob(os.path.join(directory, "*.txt")))
        if not paths:
            raise FileNotFoundError(f"No .txt label files in {directory}")
        dfs = []
        for path in paths:
            df = pd.read_csv(path, sep=None, engine="python",
                             names=["start", "end", "label"], header=None)
            stem = os.path.splitext(os.path.basename(path))[0]
            key = re.sub(r"_(FORM|TRAP)[A-Z]*$", "", stem, flags=re.IGNORECASE)
            dfs.append(df.assign(key=key))
        df = pd.concat(dfs, ignore_index=True)
        df["idx"] = df.groupby("key").cumcount()
        return df

    formal = load(formal_dir)
    trap   = load(trap_dir).rename(columns={"label": "trap_label"})

    merged = formal[["key", "idx", "start", "end", "label"]].merge(
        trap[["key", "idx", "trap_label"]],
        on=["key", "idx"], how="left"
    ).drop(columns="idx").rename(columns={"key": "song"})

    if verbose:
        n_empty = merged["trap_label"].isna().sum()
        print(f"총 {len(merged)}행, trap_label 비어 있는 행: {n_empty}")
        if n_empty:
            print("비어 있는 행이 있는 곡:",
                  merged.loc[merged["trap_label"].isna(), "song"].unique().tolist())

    return merged

In [ ]:
merged_df = merge_formal_trap(FORMAL_LABEL_DIR, TRAP_LABEL_DIR)
merged_df.head()

총 1201행, trap_label 비어 있는 행: 0


,song,start,end,label,trap_label
0,01_BLACKPINK_ASIFITSYOURLAST,0.000000,15.680000,I,I
1,01_BLACKPINK_ASIFITSYOURLAST,15.701333,31.146667,V1a,V1a
2,01_BLACKPINK_ASIFITSYOURLAST,31.168000,46.528000,V1b,V1b
3,01_BLACKPINK_ASIFITSYOURLAST,46.570667,62.933333,P1,P1
4,01_BLACKPINK_ASIFITSYOURLAST,62.954667,79.296000,C1,C1


In [ ]:
# ============================================================
# 2. 트랩 섹션 집계: 전체 트랩 곡 5곡을 뺀 22곡
# ============================================================

df = merged_df
df["is_trap"] = df["trap_label"].astype(str).str.contains("trap", case=False)

# 라벨 분해: "V2a" -> 종류 V, VCU 번호 2, 하위섹션 a
parts = df["label"].astype(str).str.extract(r"^([A-Za-z]+?)(\d*)([a-z]?)$")
df["sec_type"] = parts[0].str.upper()
df["vcu"] = parts[1]
df["section"] = parts[0].str.upper() + parts[1]          # V2a, V2b -> V2

unparsed = df.loc[parts[0].isna(), ["song", "label"]]
if len(unparsed):
    print("⚠ 형식을 인식하지 못한 라벨:\n", unparsed.drop_duplicates(), "\n")

# ------------------------------------------------------------
# (A) 두 단위로 집계
#   rows     : 라벨 파일의 행 하나 = 하나 (V2a, V2b를 따로 셈)
#   sections : 하위섹션을 합친 섹션 하나 = 하나 (V2a+V2b = V2 한 개)
#              하위섹션 중 하나라도 트랩이면 트랩 섹션으로 봄
# ------------------------------------------------------------
sec_df = (df.groupby(["song", "section"], sort=False)
            .agg(sec_type=("sec_type", "first"),
                 vcu=("vcu", "first"),
                 is_trap=("is_trap", "any"),
                 n_sub=("label", "size"))
            .reset_index())

def overview(d, unit):
    trap_songs = d.loc[d["is_trap"], "song"].unique()
    print(f"[{unit}] 전체 {d['song'].nunique()}곡 / {len(d)}개, "
          f"트랩 포함 {len(trap_songs)}곡 / 트랩 {int(d['is_trap'].sum())}개")

overview(df, "rows")
overview(sec_df, "sections")

# ------------------------------------------------------------
# (B) 곡 전체가 트랩인 곡 찾기
#   THROUGHOUT_SONGS에 5곡을 직접 적으면 그 목록을 씁니다.
#   비워두면 트랩 비율이 THRESHOLD 이상인 곡을 후보로 고릅니다.
# ------------------------------------------------------------
THROUGHOUT_SONGS = ["24_BLACKPINK_DDUDUDDUDU", "40_BTS_FAKELOVE",
                    "43_REDVELVET_BADBOY", "35_REDVELVET_PSYCHO",
                    "48_BLACKPINK_KILLTHISLOVE"]      # 예: ["03_XXX_SONGTITLE", ...]
THRESHOLD = 0.8

song_stats = (sec_df.groupby("song")
                .agg(n_sections=("section", "size"),
                     n_trap=("is_trap", "sum"))
                .assign(trap_ratio=lambda x: x["n_trap"] / x["n_sections"]))
song_stats = song_stats[song_stats["n_trap"] > 0].sort_values("trap_ratio", ascending=False)

print("\n트랩 포함 곡의 트랩 비율 (섹션 단위):")
print(song_stats.round(2).to_string())

if not THROUGHOUT_SONGS:
    THROUGHOUT_SONGS = song_stats.index[song_stats["trap_ratio"] >= THRESHOLD].tolist()
print(f"\n제외할 곡 ({len(THROUGHOUT_SONGS)}곡):", THROUGHOUT_SONGS)
if len(THROUGHOUT_SONGS) != 5:
    print("⚠ 5곡이 아닙니다. THRESHOLD를 조정하거나 THROUGHOUT_SONGS를 직접 적어주세요.")

target_songs = [s for s in song_stats.index if s not in THROUGHOUT_SONGS]
print(f"분석 대상: {len(target_songs)}곡")

# ------------------------------------------------------------
# (C) 집계: 트랩 포함 전체 곡(27곡)과 제외 후 곡(22곡)
# ------------------------------------------------------------
all_trap_songs = song_stats.index.tolist()

def summarize(d, unit, songs, title):
    sub = d[d["song"].isin(songs)]
    trap = sub[sub["is_trap"]]
    print(f"\n===== {title} [{unit}] {sub['song'].nunique()}곡 =====")
    print(f"대상 곡의 전체 {unit} 수 : {len(sub)}")
    print(f"그중 트랩 {unit} 수   : {len(trap)}")

    by_type = trap["sec_type"].value_counts()
    by_sec = trap["section"].value_counts()
    # 두 번째 버스: V2 (하위섹션 포함)
    v2 = trap[trap["section"] == "V2"]
    print(f"트랩 두 번째 버스(V2) : {len(v2)}개 / {v2['song'].nunique()}곡")
    print("\n섹션 종류별 (V, C, B ...):\n", by_type.to_string())
    print("\n섹션별 (V1, V2, C1 ...):\n", by_sec.to_string())
    return sub, trap

print("\n" + "#" * 60 + "\n# 트랩 포함 전체 곡\n" + "#" * 60)
summarize(df, "rows", all_trap_songs, "트랩 포함 전체")
summarize(sec_df, "sections", all_trap_songs, "트랩 포함 전체")

print("\n" + "#" * 60 + "\n# 전체 트랩 곡 제외\n" + "#" * 60)
rows_sub, rows_trap = summarize(df, "rows", target_songs, "제외 후")
sec_sub, sec_trap = summarize(sec_df, "sections", target_songs, "제외 후")

# 곡별 트랩 섹션 목록 (검토용)
detail = (sec_trap.groupby("song")["section"]
            .apply(lambda s: ", ".join(s)).rename("trap_sections")
            .to_frame()
            .join(song_stats[["n_sections", "n_trap"]]))
print("\n곡별 트랩 섹션:\n", detail.to_string())

# 저장 (원하면 주석 해제)
# detail.to_csv("/content/drive/MyDrive/trap_22songs_detail.csv")


# ============================================================
# 3. 바 그래프: 22곡에서 트랩 섹션의 위치 (섹션 단위)
# ============================================================
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

VERSE_COLOR = "#1f4e79"   # 버스(V) 강조
OTHER_COLOR = "#b0b0b0"

n_songs = sec_sub["song"].nunique()
n_trap = len(sec_trap)

fig, axes = plt.subplots(1, 2, figsize=(16, 6),
                         gridspec_kw={"width_ratios": [2.2, 1]})

# (a) 섹션별: V1, V2, D1 ...
by_sec = sec_trap["section"].value_counts()
colors = [VERSE_COLOR if s.startswith("V") else OTHER_COLOR for s in by_sec.index]
ax = axes[0]
bars = ax.bar(by_sec.index, by_sec.values, color=colors)
ax.bar_label(bars)
ax.set_title("(a) By section")
ax.set_xlabel("Section")
ax.set_ylabel("Count of trap-textured sections")

# (b) 섹션 종류별: V, D, B ...
by_type = sec_trap["sec_type"].value_counts()
colors = [VERSE_COLOR if t == "V" else OTHER_COLOR for t in by_type.index]
ax = axes[1]
bars = ax.bar(by_type.index, by_type.values, color=colors)
ax.bar_label(bars)
ax.set_title("(b) By section type")
ax.set_xlabel("Section type")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))

fig.suptitle(f"Count of trap texture per section "
             f"({n_songs} songs, {n_trap} trap-textured sections; "
             f"{len(THROUGHOUT_SONGS)} songs built on trap throughout excluded)")

legend_text = "\n".join(["V=verse", "P=Prechorus", "C=Chorus", "D=Drop", "B=Bridge",
                         "I=Intro", "RC=Riserchorus", "L=Link", "CO=Coda",
                         "PC=Postchorus", "1,2,3=VCU unit"])
axes[1].text(1.04, 1, legend_text, transform=axes[1].transAxes, va="top", fontsize=11)

plt.tight_layout()
# plt.savefig("/content/drive/MyDrive/trap_22songs_bar.png", dpi=300, bbox_inches="tight")
plt.show()